# Задание 6

## Часть 1 — Анализ FastQC 

---

Данные, с которыми работаем в этом задании:
`/home/STUDY/FBMF/bioinformatics/rnaseq_map_star/raw_data/Eg_Treg_S71_R1_001.fastq.gz`
`/home/STUDY/FBMF/bioinformatics/rnaseq_map_star/raw_data/Eg_Treg_S71_R2_001.fastq.gz`

1. Провел анализ файлов с помощью fastqc и объединил результаты с помощью multiqc.

Скрипт **fastqc_multiqc.slurm**:

```bash
#!/bin/bash
#SBATCH --job-name=fastqc_multiqc
#SBATCH --cpus-per-task=4
#SBATCH --mem=8gb
#SBATCH --time=00:40:00
#SBATCH --output=fastqc_multiqc_%j.log
#SBATCH --partition=AMD9554

source ~/.bashrc
conda activate bioinf

FASTQC_OUT=~/hw/hw6/results/fastqc_out
MULTIQC_OUT=~/hw/hw6/results/multiqc_out

mkdir -p $FASTQC_OUT
mkdir -p $MULTIQC_OUT

READ1=/home/STUDY/FBMF/bioinformatics/rnaseq_map_star/raw_data/Eg_Treg_S71_R1_001.fastq.gz
READ2=/home/STUDY/FBMF/bioinformatics/rnaseq_map_star/raw_data/Eg_Treg_S71_R2_001.fastq.gz

fastqc \
$READ1 \
$READ2 \
--threads 4 \
-o $FASTQC_OUT

multiqc $FASTQC_OUT -o $MULTIQC_OUT
```

2. Скачал html report в директорию гит-хаба:

```bash
scp studfbmf02_06@calc.cod.phystech.edu:/home/STUDY/FBMF/studfbmf02_06/hw/hw6/results/multiqc_out/multiqc_report.html .
```

3. **Вывод:** После просмотра отчёта можно сказать, что всё и без тримминга не плохо (об этом следующее задание)


## Часть 2 — Тримминг

---

1. Используя отчеты MultiQC из первой части, определил: 

**Вопрос:** Есть ли в ваших данных адаптеры?

**Ответ:** В конце ридов наблюдается увеличение `Adapter content`, что указывает на присутствие адаптерных последовательностей. 

![photo](pictures/hw6_1.png)

**Вопрос:** Есть ли участки с плохим качеством (Q<20) в конце ридов?

**Ответ:** Нет, там всё хорошо.

![photo](pictures/hw6_2.png)

2. Сделал тримминг ридов с помощью fastp

Скрипт **fastp_trim.slurm**

```bash
#!/bin/bash
#SBATCH --job-name=fastp_trim
#SBATCH --cpus-per-task=4
#SBATCH --mem=8gb
#SBATCH --time=00:40:00
#SBATCH --output=fastp_%j.log
#SBATCH --partition=AMD9554

source ~/.bashrc
conda activate bioinf

TRIM_OUT=~/hw/hw6/results/trimmed
FASTQC_OUT=~/hw/hw6/results/fastqc_trimmed
MULTIQC_OUT=~/hw/hw6/results/multiqc_trimmed

mkdir -p $TRIM_OUT
mkdir -p $FASTQC_OUT
mkdir -p $MULTIQC_OUT

READ1=/home/STUDY/FBMF/bioinformatics/rnaseq_map_star/raw_data/Eg_Treg_S71_R1_001.fastq.gz
READ2=/home/STUDY/FBMF/bioinformatics/rnaseq_map_star/raw_data/Eg_Treg_S71_R2_001.fastq.gz

echo "Starting fastp..."

fastp \
-i $READ1 \
-I $READ2 \
-o $TRIM_OUT/R1_trimmed.fastq.gz \
-O $TRIM_OUT/R2_trimmed.fastq.gz \
--detect_adapter_for_pe \
--cut_tail \
--cut_mean_quality 20 \
--thread 4 \
--html $TRIM_OUT/fastp.html \
--json $TRIM_OUT/fastp.json

echo "fastp completed"

echo "Starting FastQC for trimmed reads..."

fastqc \
$TRIM_OUT/R1_trimmed.fastq.gz \
$TRIM_OUT/R2_trimmed.fastq.gz \
--threads 4 \
-o $FASTQC_OUT

echo "FastQC completed"

echo "Starting MultiQC..."

multiqc $FASTQC_OUT -o $MULTIQC_OUT

echo "MultiQC completed"
```

3. Скачал отчет 

```bash
scp studfbmf02_06@calc.cod.phystech.edu:/home/STUDY/FBMF/studfbmf02_06/hw/hw6/results/multiqc_trimmed/multiqc_trim_report.html .
```

4. **Что изменилось и почему:**

- Adapter Content: адаптеры присутствовали, после тримминга линии стали ниже

![photo](pictures/hw6_1_trim.png)

- Per Base Sequence Quality: как было хорошо так и осталось 

![photo](pictures/hw6_2_trim.png)

- Sequence Length Distribution: после тримминга риды стали немного короче, так как плохие концы были обрезаны

![photo](pictures/hw6_len.png)


5. **Для дальнейшего выравнивания были выбраны:** триммированные риды, потому что

- удалены адаптерные последовательности
- уменьшено количество ошибок выравнивания


## Часть 3 — Выравнивание на референсный геном — STAR + StringTie

---

1. Выровняли риды на референсный геном `/home/STUDY/FBMF/bioinformatics/GRCh38/` с помощью `STAR`. 

Скрипт **star_mapping.slurm**

```bash
#!/bin/bash
#SBATCH --job-name=star_mapping
#SBATCH --cpus-per-task=16
#SBATCH --mem=64gb
#SBATCH --time=02:00:00
#SBATCH --output=star_%j.log
#SBATCH --partition=AMD9554

source ~/.bashrc
conda activate bioinf

GENOME_DIR="/home/STUDY/FBMF/bioinformatics/GRCh38/"
READS1="$HOME/hw/hw6/results/trimmed/R1_trimmed.fastq.gz"
READS2="$HOME/hw/hw6/results/trimmed/R2_trimmed.fastq.gz"

OUTPUT_DIR="$HOME/hw/hw6/results/star_alignment"
mkdir -p $OUTPUT_DIR

echo "Starting STAR alignment..."

STAR \
  --runThreadN 16 \
  --runMode alignReads \
  --genomeDir $GENOME_DIR \
  --readFilesIn $READS1 $READS2 \
  --readFilesCommand zcat \
  --outFileNamePrefix ${OUTPUT_DIR}/RNA_ \
  --outSAMtype BAM SortedByCoordinate \
  --quantMode GeneCounts

echo "Alignment completed"
```

2. Написал SLURM-скрипт для StringTie и запустил его на BAM-файле, полученном от STAR.

**stringtie.slurm**
```bash
#!/bin/bash
#SBATCH --job-name=stringtie
#SBATCH --cpus-per-task=4
#SBATCH --mem=32gb
#SBATCH --time=01:00:00
#SBATCH --output=stringtie_%j.log
#SBATCH --partition=AMD9554

source ~/.bashrc
conda activate bioinf

BAM="$HOME/hw/hw6/results/star_alignment/RNA_Aligned.sortedByCoord.out.bam"

OUTDIR="$HOME/hw/hw6/results/stringtie_out"

mkdir -p $OUTDIR

echo "Starting StringTie..."

stringtie \
$BAM \
-o $OUTDIR/transcripts.gtf \
-p 4

echo "StringTie completed"
```

3. **Первые 10 строк полученного GTF-файла:** 

```bash
head -n 10 ~/hw/hw6/results/stringtie_out/transcripts.gtf
# stringtie /home/STUDY/FBMF/studfbmf02_06/hw/hw6/results/star_alignment/RNA_Aligned.sortedByCoord.out.bam -o /home/STUDY/FBMF/studfbmf02_06/hw/hw6/results/stringtie_out/transcripts.gtf -p 4
# StringTie version 2.2.3
1       StringTie       transcript      75902   77168   1000    .       .       gene_id "STRG.1"; transcript_id "STRG.1.1"; cov "5.611888"; FPKM "2.244487"; TPM "4.842046";
1       StringTie       exon    75902   77168   1000    .       .       gene_id "STRG.1"; transcript_id "STRG.1.1"; exon_number "1"; cov "5.611888";
1       StringTie       transcript      91658   92659   1000    .       .       gene_id "STRG.2"; transcript_id "STRG.2.1"; cov "9.275533"; FPKM "3.709770"; TPM "8.003111";
1       StringTie       exon    91658   92659   1000    .       .       gene_id "STRG.2"; transcript_id "STRG.2.1"; exon_number "1"; cov "9.275533";
1       StringTie       transcript      94864   96518   1000    .       .       gene_id "STRG.3"; transcript_id "STRG.3.1"; cov "4.845505"; FPKM "1.937971"; TPM "4.180796";
1       StringTie       exon    94864   96518   1000    .       .       gene_id "STRG.3"; transcript_id "STRG.3.1"; exon_number "1"; cov "4.845505";
1       StringTie       transcript      99112   102057  1000    .       .       gene_id "STRG.4"; transcript_id "STRG.4.1"; cov "4.882952"; FPKM "1.952948"; TPM "4.213106";
1       StringTie       exon    99112   102057  1000    .       .       gene_id "STRG.4"; transcript_id "STRG.4.1"; exon_number "1"; cov "4.882952";
```

**Какие поля там присутствуют и что означают:**
- `gene_id`: ID гена
- `transcript_id`: ID транскрипта
- `cov`: среднее покрытие транскрипта ридами (coverage). Показывает, насколько хорошо данный участок был покрыт RNA-seq ридами.
- `FPKM`: нормализованная оценка экспрессии транскрипта (Fragments Per Kilobase of transcript per Million mapped reads). Учитывает длину транскрипта и общее число ридов в образце.
- `TPM`: нормализованная abundance транскрипта (Transcripts Per Million). Показывает относительный уровень экспрессии транскрипта и удобен для сравнения между образцами.


**Сколько всего транскриптов получилось:**

```bash 
grep -c "transcript" ~/hw/hw6/results/stringtie_out/transcripts.gtf
183959
```

`183959` транскриптов получилось 

**Что даёт сборка транскриптов StringTie, для каких задач полезен transcripts.gtf:**

`StringTie` выполняет сборку транскриптов по RNA-seq данным и создаёт файл `transcripts.gtf` с аннотацией найденных транскриптов. 

Этот файл полезен для анализа экспрессии генов, поиска новых транскриптов и изоформ, исследования альтернативного сплайсинга и дальнейшего RNA-seq анализа

## Часть 4 — Выравнивание на референсный геном — HTSeq

---

1. Выровнял риды на референсный геном `/home/STUDY/FBMF/bioinformatics/GRCh38/` с помощью `HTSeq`. 

**htseq_count.slurm**
```bash
#!/bin/bash
#SBATCH --job-name=htseq_count
#SBATCH --cpus-per-task=2
#SBATCH --mem=8gb
#SBATCH --time=01:00:00
#SBATCH --output=htseq_%j.log
#SBATCH --partition=AMD9554

source ~/.bashrc
conda activate rnaseq

BAM="$HOME/hw/hw6/results/star_alignment/RNA_Aligned.sortedByCoord.out.bam"

GTF="/home/STUDY/FBMF/bioinformatics/GRCh38/Homo_sapiens.GRCh38.110.chr.gtf"

OUT="$HOME/hw/hw6/results/htseq_counts.txt"

echo "Starting HTSeq-count..."

htseq-count \
    --format bam \
    --order pos \
    --stranded no \
    --type exon \
    --idattr gene_id \
    $BAM $GTF > $OUT

echo "HTSeq-count completed"
```

2. **Сколько уникальных генов получили в htseq_counts.txt:**

```bash
grep -v "^__" ~/hw/hw6/results/htseq_counts.txt | wc -l
62700
```

`62700` уникальных генов получилось

## Выводы

--- 

**1. Логические шаги в задании:** Сначала данные были проверены на качество с помощью FastQC и объединены в общий отчёт MultiQC для оценки адаптеров и распределения качества ридов. Затем был выполнен тримминг ридов с помощью fastp для удаления адаптеров и низкокачественных участков, после чего анализ качества был повторён

Далее было выполнено выравнивание очищенных ридов на геном, в результате чего получены BAM-файлы и таблицы экспрессии генов. На основе выравнивания были выполнены две разные оценки экспрессии: сборка транскриптов с помощью StringTie и подсчёт ридов по генам с помощью HTSeq

**2. Скриншот вашей папки с заданием на сервере, где все файлы выравнивания** (привожу вывод команды, так как в скриншоте плохо видно)

```bash
ls -lhR ~/hw/hw6/results
/home/STUDY/FBMF/studfbmf02_06/hw/hw6/results:
total 1.2M
drwxr-xr-x. 2 studfbmf02_06 fbmf    4 May  7 16:28 fastqc_out
drwxr-xr-x. 2 studfbmf02_06 fbmf    4 May  7 16:46 fastqc_trimmed
-rw-r--r--. 1 studfbmf02_06 fbmf 1.2M May 11 11:39 htseq_counts.txt
drwxr-xr-x. 3 studfbmf02_06 fbmf    2 May  7 16:28 multiqc_out
drwxr-xr-x. 3 studfbmf02_06 fbmf    2 May  7 16:51 multiqc_trimmed
drwxr-xr-x. 2 studfbmf02_06 fbmf    6 May 10 20:43 star_alignment
drwxr-xr-x. 2 studfbmf02_06 fbmf    1 May 11 11:05 stringtie_out
drwxr-xr-x. 2 studfbmf02_06 fbmf    4 May  7 16:45 trimmed

/home/STUDY/FBMF/studfbmf02_06/hw/hw6/results/fastqc_out:
total 2.2M
-rw-r--r--. 1 studfbmf02_06 fbmf 603K May  7 16:28 Eg_Treg_S71_R1_001_fastqc.html
-rw-r--r--. 1 studfbmf02_06 fbmf 478K May  7 16:28 Eg_Treg_S71_R1_001_fastqc.zip
-rw-r--r--. 1 studfbmf02_06 fbmf 606K May  7 16:28 Eg_Treg_S71_R2_001_fastqc.html
-rw-r--r--. 1 studfbmf02_06 fbmf 485K May  7 16:28 Eg_Treg_S71_R2_001_fastqc.zip

/home/STUDY/FBMF/studfbmf02_06/hw/hw6/results/fastqc_trimmed:
total 2.2M
-rw-r--r--. 1 studfbmf02_06 fbmf 622K May  7 16:46 R1_trimmed_fastqc.html
-rw-r--r--. 1 studfbmf02_06 fbmf 464K May  7 16:46 R1_trimmed_fastqc.zip
-rw-r--r--. 1 studfbmf02_06 fbmf 628K May  7 16:46 R2_trimmed_fastqc.html
-rw-r--r--. 1 studfbmf02_06 fbmf 474K May  7 16:46 R2_trimmed_fastqc.zip

/home/STUDY/FBMF/studfbmf02_06/hw/hw6/results/multiqc_out:
total 1.1M
drwxr-xr-x. 2 studfbmf02_06 fbmf    4 May  7 16:28 multiqc_data
-rw-r--r--. 1 studfbmf02_06 fbmf 1.1M May  7 16:28 multiqc_report.html

/home/STUDY/FBMF/studfbmf02_06/hw/hw6/results/multiqc_out/multiqc_data:
total 12K
-rw-r--r--. 1 studfbmf02_06 fbmf  831 May  7 16:28 multiqc_fastqc.txt
-rw-r--r--. 1 studfbmf02_06 fbmf  257 May  7 16:28 multiqc_general_stats.txt
-rw-r--r--. 1 studfbmf02_06 fbmf 9.4K May  7 16:28 multiqc.log
-rw-r--r--. 1 studfbmf02_06 fbmf  286 May  7 16:28 multiqc_sources.txt

/home/STUDY/FBMF/studfbmf02_06/hw/hw6/results/multiqc_trimmed:
total 1.1M
drwxr-xr-x. 2 studfbmf02_06 fbmf    4 May  7 16:46 multiqc_data
-rw-r--r--. 1 studfbmf02_06 fbmf 1.1M May  7 16:46 multiqc_trim_report.html

/home/STUDY/FBMF/studfbmf02_06/hw/hw6/results/multiqc_trimmed/multiqc_data:
total 12K
-rw-r--r--. 1 studfbmf02_06 fbmf  804 May  7 16:46 multiqc_fastqc.txt
-rw-r--r--. 1 studfbmf02_06 fbmf  244 May  7 16:46 multiqc_general_stats.txt
-rw-r--r--. 1 studfbmf02_06 fbmf 9.5K May  7 16:46 multiqc.log
-rw-r--r--. 1 studfbmf02_06 fbmf  246 May  7 16:46 multiqc_sources.txt

/home/STUDY/FBMF/studfbmf02_06/hw/hw6/results/star_alignment:
total 2.0G
-rw-r--r--. 1 studfbmf02_06 fbmf 2.0G May 10 20:43 RNA_Aligned.sortedByCoord.out.bam
-rw-r--r--. 1 studfbmf02_06 fbmf 2.0K May 10 20:43 RNA_Log.final.out
-rw-r--r--. 1 studfbmf02_06 fbmf  15K May 10 20:43 RNA_Log.out
-rw-r--r--. 1 studfbmf02_06 fbmf  600 May 10 20:43 RNA_Log.progress.out
-rw-r--r--. 1 studfbmf02_06 fbmf 1.4M May 10 20:43 RNA_ReadsPerGene.out.tab
-rw-r--r--. 1 studfbmf02_06 fbmf 4.5M May 10 20:43 RNA_SJ.out.tab

/home/STUDY/FBMF/studfbmf02_06/hw/hw6/results/stringtie_out:
total 25M
-rw-r--r--. 1 studfbmf02_06 fbmf 25M May 11 11:05 transcripts.gtf

/home/STUDY/FBMF/studfbmf02_06/hw/hw6/results/trimmed:
total 1.8G
-rw-r--r--. 1 studfbmf02_06 fbmf 442K May  7 16:45 fastp.html
-rw-r--r--. 1 studfbmf02_06 fbmf 109K May  7 16:45 fastp.json
-rw-r--r--. 1 studfbmf02_06 fbmf 814M May  7 16:45 R1_trimmed.fastq.gz
-rw-r--r--. 1 studfbmf02_06 fbmf 935M May  7 16:45 R2_trimmed.fastq.gz
```

**3. Сравнение каунтов из STAR (RNA_ReadsPerGene.out.tab, колонка 3) с HTSeq‑count**

Построил scatter plot в Python (скрипт ниже).

**compare_counts.py**
```python
import pandas as pd
import matplotlib.pyplot as plt

BASE = "/home/STUDY/FBMF/studfbmf02_06/hw/hw6/results"
star_path = f"{BASE}/star_alignment/RNA_ReadsPerGene.out.tab"
htseq_path = f"{BASE}/htseq_counts.txt"

star = pd.read_csv(star_path, sep="\t", header=None)

star = star.iloc[:, [0, 2]]
star.columns = ["gene", "star_count"]

htseq = pd.read_csv(htseq_path, sep="\t", header=None,
                    names=["gene", "htseq_count"])

htseq = htseq[~htseq["gene"].str.startswith("__")]

df = pd.merge(star, htseq, on="gene", how="inner")

plt.figure(figsize=(6,6))
plt.scatter(df["star_count"], df["htseq_count"], alpha=0.5)

plt.xlabel("STAR counts")
plt.ylabel("HTSeq counts")
plt.title("STAR vs HTSeq gene counts")

plt.xscale("log")
plt.yscale("log")

plt.savefig("star_vs_htseq.png", dpi=300)
```

![photo](pictures/star_vs_htseq.png)

**Корреляция:**

- На графике наблюдается сильная линейная зависимость между подсчетами `STAR` и `HTSeq`. Точки плотно сгруппированы вдоль диагональной линии, что говорит о высокой согласованности результатов двух методов для большинства генов.

- Видно, что при низкой экспрессии разброс точек увеличивается. Это означает, что для генов с малым количеством прочтений методы чаще дают немного разные результаты из-за большей чувствительности к случайным флуктуациям и разницы в алгоритмах отсева шумов

**Что лучше использовать для дальнейшего DE‑анализа (DESeq2/edgeR) – каунты STAR или HTSeq?**

Лучше использовать каунты `HTSeq`. Они более консервативны и точны на уровне генов, так как строже работают с неоднозначно картированными ридами
